# 02 - Pipeline de Carga (Data Load) no Azure SQL Server
**Squad 2 — Real Time for Business | Dupla 1**  
**Integrantes:** Lucas Sousa Santos Oliveira & Zaiden Emiliano Segundo Seleme  
**Tabelas:** `ecommerce_produtos` e `ecommerce_categorias`  
**Tabelas Destino:** `squad2.ecommerce_produtos` e `squad2.ecommerce_categorias`  

### Objetivo:
Executar a ingestão e gravação dos dados consolidados de tempo real no schema `squad2` do Azure SQL Server utilizando o conector nativo `sqlserver`.

**Ordem no pipeline:** Executar após `01_extracao_produtos_categorias.ipynb` e antes de `03_auditoria_data_quality.ipynb`.

In [0]:
import os
import io
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

load_dotenv(find_dotenv(), override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

sql_host = os.getenv("SQL_HOST", "srv-database-intership.database.windows.net")
sql_database = os.getenv("SQL_DATABASE", "internshipDatabase")
sql_user = os.getenv("SQL_USERNAME", "estagiario_user")
sql_password = os.getenv("SQL_PASSWORD")

jdbc_url = f"jdbc:sqlserver://{sql_host}:1433;database={sql_database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
jdbc_driver = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

credential = ClientSecretCredential(tenant_id, client_id, client_secret)
service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account}.dfs.core.windows.net",
    credential=credential
)
fs_raw = service_client.get_file_system_client("raw")
print("Credenciais carregadas e conexão estabelecida.")

## 1. Extração e Consolidação das Janelas de Tempo Real

In [0]:
# Dicionário de opções OAuth para leitura direta no cluster Spark
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Leitura distribuída nativa no cluster Spark (sem trafegar no driver)
caminho_prod = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data/*/*/*/*/ecommerce_produtos.parquet"
caminho_cat = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data/*/*/*/*/ecommerce_categorias.parquet"

print("Lendo microbatches de tempo real via cluster Spark...")
df_produtos_raw = spark.read.options(**adls_options).parquet(caminho_prod)
df_categorias_raw = spark.read.options(**adls_options).parquet(caminho_cat)

# Deduplicação distribuída
df_produtos = df_produtos_raw.dropDuplicates(["sku"])
df_categorias = df_categorias_raw.dropDuplicates(["id_categoria"])

print(f"Produtos prontos para carga: {df_produtos.count()} registros únicos.")
print(f"Categorias prontas para carga: {df_categorias.count()} registros únicos.")

## 2. Execução da Carga no Azure SQL Server (`squad2`)

In [0]:
def salvar_no_sql(df, nome_tabela_destino):
    print(f"Iniciando carga na tabela '{nome_tabela_destino}'...")
    
    # 1. Tentativa com o conector nativo "sqlserver" exigido pelo Databricks Serverless
    try:
        print("  -> Gravando via conector nativo 'sqlserver'...")
        df.write \
            .format("sqlserver") \
            .option("host", sql_host) \
            .option("port", "1433") \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .option("database", sql_database) \
            .option("dbtable", nome_tabela_destino) \
            .mode("overwrite") \
            .save()
        print(f"Status da Carga: Sucesso via conector 'sqlserver' na tabela '{nome_tabela_destino}'!")
        return
    except Exception as e_sqlserver:
        print(f"  Aviso: conector nativo 'sqlserver' retornou: {e_sqlserver}")
        
    # 2. Tentativa com format("sqlserver") passando URL
    try:
        print("  -> Tentando via conector 'sqlserver' com JDBC URL...")
        df.write \
            .format("sqlserver") \
            .option("url", jdbc_url) \
            .option("dbtable", nome_tabela_destino) \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .mode("overwrite") \
            .save()
        print(f"Status da Carga: Sucesso na tabela '{nome_tabela_destino}'!")
        return
    except Exception as e_url:
        print(f"  Aviso: tentativa com URL retornou: {e_url}")

    # 3. Fallback inteligente via pyodbc (Bypass para Serverless caso Spark Connect bloqueie DML)
    print("  -> Ativando bypass estratégico de carga via pyodbc...")
    import pyodbc
    pdf = df.toPandas()
    conn_str = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={sql_host};DATABASE={sql_database};UID={sql_user};PWD={sql_password};Encrypt=yes;TrustServerCertificate=no;"
    try:
        conn = pyodbc.connect(conn_str, timeout=30)
    except Exception:
        # Fallback de driver ODBC comum
        conn_str_legacy = f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={sql_host};DATABASE={sql_database};UID={sql_user};PWD={sql_password};Encrypt=yes;TrustServerCertificate=no;"
        conn = pyodbc.connect(conn_str_legacy, timeout=30)
        
    cursor = conn.cursor()
    cursor.fast_executemany = True
    
    # Criar tabela se não existir
    colunas_sql = []
    for col, dtype in pdf.dtypes.items():
        if "int" in str(dtype):
            colunas_sql.append(f"[{col}] BIGINT")
        elif "float" in str(dtype):
            colunas_sql.append(f"[{col}] FLOAT")
        elif "bool" in str(dtype):
            colunas_sql.append(f"[{col}] BIT")
        else:
            colunas_sql.append(f"[{col}] NVARCHAR(MAX)")
            
    schema_name, table_name = nome_tabela_destino.split(".")
    cursor.execute(f"IF OBJECT_ID('{nome_tabela_destino}', 'U') IS NOT NULL DROP TABLE {nome_tabela_destino};")
    cursor.execute(f"CREATE TABLE {nome_tabela_destino} ({', '.join(colunas_sql)});")
    conn.commit()
    
    # Inserção em massa
    placeholders = ", ".join(["?"] * len(pdf.columns))
    insert_sql = f"INSERT INTO {nome_tabela_destino} VALUES ({placeholders})"
    # Converte nans para None
    dados_inserir = [tuple(None if pd.isna(v) else v for v in row) for row in pdf.itertuples(index=False)]
    cursor.executemany(insert_sql, dados_inserir)
    conn.commit()
    conn.close()
    print(f"Status da Carga: Sucesso via pyodbc bypass na tabela '{nome_tabela_destino}' ({len(pdf)} registros gravados)!")
# Execução da carga das tabelas no SQL Server
salvar_no_sql(df_produtos, "squad2.ecommerce_produtos")
salvar_no_sql(df_categorias, "squad2.ecommerce_categorias")


## 3. Validação Imediata da Carga

In [0]:
def validar_carga(nome_tabela_destino):
    print(f"Validando leitura de '{nome_tabela_destino}'...")
    try:
        df_val = spark.read \
            .format("sqlserver") \
            .option("host", sql_host) \
            .option("port", "1433") \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .option("database", sql_database) \
            .option("dbtable", nome_tabela_destino) \
            .load()
    except Exception:
        df_val = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", nome_tabela_destino) \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .option("driver", jdbc_driver) \
            .load()
            
    qtd = df_val.count()
    print(f"Validação {nome_tabela_destino}: {qtd} registros confirmados no SQL Server!")
    return df_val

validar_carga("squad2.ecommerce_produtos")
validar_carga("squad2.ecommerce_categorias")

print("\nPipeline de carga finalizado com sucesso!")